# Ch14. Neural Networks
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 5] 14.2 MLP for Air Passengers

In [ ]:
test_mask = AirPassengersPanel["ds"] >= "1960"
Y_train_df = AirPassengersPanel[~test_mask]
Y_test_df  = AirPassengersPanel[test_mask].reset_index(drop=True)

model = MLP(
    h=12,            # forecast horizon
    input_size=24,   # 24 months of history
    scaler_type="robust",
)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

## [Slide 8] 14.3 NHITS and RNN Example

In [ ]:
models = [
    NHITS(h=12, input_size=24, scaler_type="robust"),
    RNN(h=12,   input_size=24, scaler_type="robust"),
]
nf = NeuralForecast(models=models, freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

fig, axes = plt.subplots(2, sharex=True, figsize=(8, 5.5))
plot_series(
    AirPassengersPanel, forecasts,
    xlabel="", ylabel="",
    palette="black_and_2color", rm_legend=False,
    ax=axes, legend_loc="outside lower center",
)
axes[-1].set(xlabel="Month [1M]")
fig.suptitle("Number of passengers for different airlines",
             x=.54, fontsize=12)
fig.supylabel("Passengers")

## [Slide 10] 14.4 Scaling the Data

In [ ]:
# Poor performance without scaling:
model = MLP(h=12, input_size=24, scaler_type="identity")

## [Slide 12] 14.5 Optimisation Objectives

In [ ]:
model = NHITS(h=12, input_size=24,
              loss=MSE(), scaler_type="robust")

## [Slide 14] 14.5 Probabilistic Optimisation Objectives

In [ ]:
model = NHITS(
    h=12,
    input_size=24,
    loss=DistributionLoss(distribution="Normal"),
    scaler_type="robust",
)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
# Produces prediction intervals (80%, 90% by default)
forecasts = nf.predict()

## [Slide 16] 14.6 Exogenous Variables

In [ ]:
df        = pd.read_csv("data/EPF_FR_BE.csv", parse_dates=["ds"])
static_df = pd.read_csv("data/EPF_FR_BE_static.csv")
futr_df   = pd.read_csv("data/EPF_FR_BE_futr.csv",
                         parse_dates=["ds"])

## [Slide 18] 14.6 BiTCN with Exogenous Variables

In [ ]:
horizon = 24   # day-ahead hourly forecast

model = BiTCN(
    h=horizon,
    input_size=5 * horizon,
    futr_exog_list=["gen_forecast", "week_day"],
    hist_exog_list=["system_load"],
    stat_exog_list=["market_0", "market_1"],
    scaler_type="robust",
    max_steps=300,
)
nf = NeuralForecast(models=[model], freq="H")
nf.fit(df=df, static_df=static_df)
forecasts = nf.predict(futr_df=futr_df)

## [Slide 20] 14.7 Hyperparameter Optimisation

In [ ]:
from ray import tune

nhits_config = {
    **AutoNHITS.get_default_config(h=12, backend="ray"),
    "random_seed": tune.randint(1, 10),
    "n_pool_kernel_size": tune.choice(
        [[2, 2, 2], [16, 8, 1]]
    ),
    "max_steps": tune.choice([100]),
}
model = AutoNHITS(h=12, num_samples=10,
                  config=nhits_config)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()

## [Slide 22] 14.7 Cross-Validation for Neural Networks

In [ ]:
nf = NeuralForecast(
    models=[NHITS(h=12, input_size=24,
                  loss=MAE(), max_steps=500)],
    freq="MS"
)
# Time-series cross-validation with n_windows expanding windows
cv_df = nf.cross_validation(
    df=Y_train_df,
    n_windows=3,
    step_size=12,
)